
# AI IN HEALTHCARE: High Risk Project COLAB NOTEBOOK
# Mohsin Imam
# Topic: Translator


In [11]:
!pip install -q -U google-generativeai pandas

In [13]:
import pandas as pd
import google.generativeai as genai
import json
import time
from google.colab import userdata

# --- 1. SETUP GEMINI API ---
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
# We use JSON mode to ensure the output is perfectly structured for our training dataset
model = genai.GenerativeModel(
    'gemini-1.5-flash',
    generation_config={"response_mime_type": "application/json"}
)

# --- 2. LOAD SYNTHEA DATA ---
print("Loading Synthea CSVs...")
try:
    patients_df = pd.read_csv('patients.csv')
    conditions_df = pd.read_csv('conditions.csv')
    medications_df = pd.read_csv('medications.csv')
except FileNotFoundError:
    print("Error: Please make sure patients.csv, conditions.csv, and medications.csv are uploaded to Colab.")

# --- 3. STITCH TABULAR DATA INTO CLINICAL NOTES ---
print("Generating synthetic clinical notes...")
patient_ids = patients_df['Id'].head(100).tolist()
synthetic_notes = []

for pid in patient_ids:
    # Extract unique conditions and medications for the specific patient
    pt_conditions = conditions_df[conditions_df['PATIENT'] == pid]['DESCRIPTION'].dropna().unique().tolist()
    pt_meds = medications_df[medications_df['PATIENT'] == pid]['DESCRIPTION'].dropna().unique().tolist()

    # Skip patients with no medical history to ensure our training data is robust
    if not pt_conditions:
        continue

    # Construct the jargon-heavy note
    note = f"Patient ID {pid[:8]} presents with a documented medical history significant for {', '.join(pt_conditions)}. "
    if pt_meds:
        note += f"Current pharmacological interventions include: {', '.join(pt_meds)}. "
    note += "Plan: Continue current care regimen, monitor for therapeutic efficacy and adverse interactions. Follow up in outpatient clinic."

    synthetic_notes.append(note)

notes_df = pd.DataFrame({'clinical_note': synthetic_notes})
print(f"Successfully generated {len(notes_df)} clinical notes.\n")

# --- 4. GENERATE GROUND TRUTH (TEACHER MODEL) ---
print("Calling Gemini API to generate English & Urdu ground truth...")
training_data = []

# We'll process a small batch first to ensure everything works perfectly
for index, row in notes_df.iterrows():
    print(f"Translating note {index + 1}/{len(notes_df)}...")

    prompt = f"""
    You are an expert medical translator. Read the following clinical note and output exactly two things:
    1. "simple_english": A simplified summary written at an 8th-grade reading level, removing complex jargon but keeping clinical accuracy.
    2. "urdu_translation": A highly accurate, culturally appropriate Urdu translation of the simplified English summary.

    Clinical Note: {row['clinical_note']}
    """

    try:
        response = model.generate_content(prompt)
        result = json.loads(response.text)

        # Structure it exactly how the Gemini Fine-Tuning API expects it
        training_data.append({
            "text_input": row['clinical_note'],
            "output": f"Simple English: {result['simple_english']}\n\nUrdu: {result['urdu_translation']}"
        })

        # Sleep to respect API rate limits
        time.sleep(3)

    except Exception as e:
        print(f"Error processing note {index + 1}: {e}")

# --- 5. SAVE DATASET ---
# Convert the training data dictionary into a DataFrame to inspect and save
training_df = pd.DataFrame(training_data)
training_df.to_csv('fine_tuning_dataset.csv', index=False)

print("\nPipeline Complete! Here is a preview of your training data:")
print(training_df.head())

TimeoutException: Requesting secret GEMINI_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.